# 18C — Fannie multi-quarter schema and population audit
Audit the five pilot acquisition archives before selecting any loans. This stage checks physical structure and date/key consistency; semantic compatibility and coverage remain subject to review. No raw files are changed.

Revision: audit the current 114-field release separately. The glossary supports the early date and ID positions; the complete physical predictor layout and exact release consistency remain unverified. Do not sample or construct targets yet.

## 1. Locate pilot archives and recover reviewed field positions

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib, json, re, time, zipfile, ctypes
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
RAW = ROOT / 'data/raw/Fannie_Mae/release_2026Q2'
EXPECTED_FIELDS = 114
quarters = ['2018Q1', '2018Q2', '2018Q3', '2018Q4', '2019Q1']
for q in quarters:
    assert (RAW / f'{q}.zip').is_file(), f'Missing {q}.zip'
class MemoryCounters(ctypes.Structure):
    _fields_ = [('cb', ctypes.c_ulong), ('PageFaultCount', ctypes.c_ulong)] + [(name, ctypes.c_size_t) for name in ['PeakWorkingSetSize','WorkingSetSize','QuotaPeakPagedPoolUsage','QuotaPagedPoolUsage','QuotaPeakNonPagedPoolUsage','QuotaNonPagedPoolUsage','PagefileUsage','PeakPagefileUsage']]

def peak_ram_gib():
    try:
        p = MemoryCounters()
        p.cb = ctypes.sizeof(p)
        ctypes.windll.kernel32.GetCurrentProcess.restype = ctypes.c_void_p
        handle = ctypes.windll.kernel32.GetCurrentProcess()
        ok = ctypes.windll.psapi.GetProcessMemoryInfo(ctypes.c_void_p(handle), ctypes.byref(p), p.cb)
        return round(p.PeakWorkingSetSize / 2**30, 4) if ok else None
    except AttributeError:
        return None

# Confirm the origination field position against the reviewed Notebook 15 code.
import ast
source_notebook = ROOT / 'notebooks/15_fannie_external_dataset_construction.ipynb'
assert source_notebook.is_file(), 'Notebook 15 is required to recover its reviewed field positions.'
positions = None
for cell in json.loads(source_notebook.read_text(encoding='utf-8'))['cells']:
    if cell['cell_type'] != 'code':
        continue
    try:
        tree = ast.parse(''.join(cell['source']))
    except SyntaxError:
        continue
    for node in ast.walk(tree):
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == 'POSITIONS' for t in node.targets):
            positions = ast.literal_eval(node.value)
assert isinstance(positions, dict) and 'orig' in positions, 'Review POSITIONS in Notebook 15.'
orig_index = positions['orig'] - 1
assert orig_index == 13, 'Glossary origination position differs from Notebook 15.'
print('Reviewed origination position:', orig_index + 1)
assert 'report' in positions and 'first_payment' in positions
report_index = positions['report'] - 1
payment_index = positions['first_payment'] - 1
assert (report_index, payment_index) == (2, 14), 'Glossary date positions differ from Notebook 15.'
print('Reviewed positions:', {k: positions[k] for k in ['orig','report','first_payment']})

# Preflight every archive before the full scan.
for q in quarters:
    with zipfile.ZipFile(RAW / f'{q}.zip') as z:
        members = [m for m in z.infolist() if not m.is_dir() and m.filename.lower().endswith('.csv')]
        assert len(members) == 1, f'{q}: unexpected archive contents'
        with z.open(members[0]) as stream:
            sample = [stream.readline() for _ in range(100)]
        counts = Counter(len(r.rstrip(b'\r\n').split(b'|')) for r in sample if r)
        print(q, dict(counts))
        assert counts == {EXPECTED_FIELDS: 100}, f'{q}: unexpected layout; review before scanning'


Reviewed origination position: 14
Reviewed positions: {'orig': 14, 'report': 3, 'first_payment': 15}
2018Q1 {114: 100}
2018Q2 {114: 100}
2018Q3 {114: 100}
2018Q4 {114: 100}
2019Q1 {114: 100}


## 2. Scan each ZIP
Read each archive sequentially, verify all physical records, and count unique loans. Origination conflicts are flagged rather than resolved. Reporting and first-payment date problems are counted, not silently removed. Cross-file overlaps are counted using unique identifiers in memory; identifiers are never exported.

In [2]:
date_pattern = re.compile(rb'(0[1-9]|1[0-2])[0-9]{4}')
id_pattern = re.compile(rb'[A-Za-z0-9]{1,12}')

def valid_date(value):
    return bool(date_pattern.fullmatch(value)) and 1900 <= int(value[2:]) <= 2100

def scan_archive(q, prior_ids):
    path = RAW / f'{q}.zip'
    signature = (path.stat().st_size, path.stat().st_mtime_ns)
    counts, quality = Counter(), Counter()
    seen = set()
    current = None
    dates, payments = set(), set()
    report_min, report_max = None, None
    sha = hashlib.sha256()
    start = time.perf_counter()
    rows = 0

    def finish(key, dates, payments):
        if key is None:
            return
        if key in seen:
            raise ValueError(f'{q}: noncontiguous loan history; stop for review')
        seen.add(key)
        if len(dates) != 1:
            counts['conflicting_or_missing'] += 1
        else:
            value = next(iter(dates))
            if not valid_date(value):
                counts['invalid_date'] += 1
            else:
                counts[f'{value[2:].decode()}Q{(int(value[:2])-1)//3+1}'] += 1
        if len(payments) != 1 or not valid_date(next(iter(payments), b'')):
            quality['loan_first_payment_unresolved'] += 1

    with zipfile.ZipFile(path) as z:
        members = [m for m in z.infolist() if not m.is_dir() and m.filename.lower().endswith('.csv')]
        assert len(members) == 1, f'{q}: review CSV archive members'
        member = members[0]
        assert Path(member.filename).name.lower() == f'{q.lower()}.csv', f'{q}: unexpected CSV name'
        with z.open(member) as stream:
            for rows, raw in enumerate(stream, 1):
                sha.update(raw)
                parts = raw.rstrip(b'\r\n').split(b'|')
                if len(parts) != EXPECTED_FIELDS:
                    raise ValueError(f'{q}: expected {EXPECTED_FIELDS} fields, found {len(parts)} at row {rows}')
                key = parts[1].strip()
                if not id_pattern.fullmatch(key):
                    raise ValueError(f'{q}: invalid loan ID at row {rows}')
                if key != current:
                    finish(current, dates, payments)
                    current, dates, payments = key, set(), set()
                orig, payment, report = (parts[i].strip() for i in [orig_index, payment_index, report_index])
                if orig: dates.add(orig)
                if payment: payments.add(payment)
                if not valid_date(orig): quality['row_origination_missing_or_invalid'] += 1
                if not valid_date(payment): quality['row_first_payment_missing_or_invalid'] += 1
                if not valid_date(report):
                    quality['row_report_missing_or_invalid'] += 1
                else:
                    period = int(report[2:]) * 100 + int(report[:2])
                    report_min = period if report_min is None else min(period, report_min)
                    report_max = period if report_max is None else max(period, report_max)
                if rows % 5_000_000 == 0:
                    print(f'{q}: {rows:,} rows scanned', flush=True)
            finish(current, dates, payments)
    assert rows > 0 and sum(counts.values()) == len(seen)
    assert signature == (path.stat().st_size, path.stat().st_mtime_ns)
    overlap = len(seen & prior_ids)
    prior_ids.update(seen)
    result = dict(acquisition_file=q, zip_bytes=signature[0], csv_member=member.filename,
                  csv_bytes=member.file_size, physical_rows=rows, unique_loans=len(seen),
                  overlap_with_previous_files=overlap, fields_per_row=EXPECTED_FIELDS,
                  earliest_report=report_min, latest_report=report_max,
                  decompressed_sha256=sha.hexdigest(), scan_seconds=round(time.perf_counter()-start,2),
                  process_peak_working_set_gib=peak_ram_gib())
    print(q, result['unique_loans'], 'loans;', overlap, 'overlaps;', result['scan_seconds'], 'seconds', flush=True)
    return result, counts, quality

prior_ids = set()
inventory, distribution, checks = [], [], []
for q in quarters:
    result, counts, quality = scan_archive(q, prior_ids)
    inventory.append(result)
    distribution.extend(dict(acquisition_file=q, origination_cohort=k, loans=v) for k,v in sorted(counts.items()))
    checks.extend(dict(acquisition_file=q, check=k, count=quality[k]) for k in ['loan_first_payment_unresolved','row_origination_missing_or_invalid','row_first_payment_missing_or_invalid','row_report_missing_or_invalid'])


2018Q1: 5,000,000 rows scanned
2018Q1: 10,000,000 rows scanned
2018Q1: 15,000,000 rows scanned
2018Q1: 20,000,000 rows scanned
2018Q1 458830 loans; 0 overlaps; 267.82 seconds
2018Q2: 5,000,000 rows scanned
2018Q2: 10,000,000 rows scanned
2018Q2: 15,000,000 rows scanned
2018Q2: 20,000,000 rows scanned
2018Q2 454581 loans; 0 overlaps; 270.28 seconds
2018Q3: 5,000,000 rows scanned
2018Q3: 10,000,000 rows scanned
2018Q3: 15,000,000 rows scanned
2018Q3: 20,000,000 rows scanned
2018Q3 500018 loans; 0 overlaps; 262.62 seconds
2018Q4: 5,000,000 rows scanned
2018Q4: 10,000,000 rows scanned
2018Q4: 15,000,000 rows scanned
2018Q4 421040 loans; 0 overlaps; 236.12 seconds
2019Q1: 5,000,000 rows scanned
2019Q1: 10,000,000 rows scanned
2019Q1 341865 loans; 0 overlaps; 163.52 seconds


## 3. Review acquisition coverage
Counts are per-file and are not deduplicated across files. If overlap is nonzero, reconcile the underlying histories before sampling. The 2019 Q1 file tests delayed acquisitions; it cannot establish complete coverage by itself.

In [3]:
inventory_df = pd.DataFrame(inventory)
cohorts = pd.DataFrame(distribution)
quality_df = pd.DataFrame(checks)
coverage = cohorts[cohorts.origination_cohort.str.fullmatch(r'2018Q[1-4]')].pivot_table(index='acquisition_file',columns='origination_cohort',values='loans',aggfunc='sum',fill_value=0).reindex(quarters,fill_value=0).reindex(columns=['2018Q1','2018Q2','2018Q3','2018Q4'],fill_value=0)
coverage['2018_total'] = coverage.sum(axis=1)
assert inventory_df.unique_loans.sum() - inventory_df.overlap_with_previous_files.sum() == len(prior_ids)
display(inventory_df.drop(columns=['decompressed_sha256','csv_member']))
display(coverage)
display(quality_df)
print('Total distinct IDs across files:', len(prior_ids))
print('2018 counts remain provisional; assess later acquisitions and release consistency.')

old_rows, old_loans = 23563399, 458830
q1 = inventory_df.loc[inventory_df.acquisition_file.eq('2018Q1')].iloc[0]
old_hash = 'b89043e7c2967d8f50bb3d7f5f694ee624a77ec86e9390a6333ee9ccf5ed3ebe'
reconciliation = pd.DataFrame([
    {'measure':'physical_rows','original':old_rows,'current':int(q1.physical_rows),'difference':int(q1.physical_rows)-old_rows},
    {'measure':'unique_loans','original':old_loans,'current':int(q1.unique_loans),'difference':int(q1.unique_loans)-old_loans},
    {'measure':'fields_per_row','original':113,'current':int(q1.fields_per_row),'difference':int(q1.fields_per_row)-113},
])
display(reconciliation)
print('Same decompressed hash as original:', q1.decompressed_sha256 == old_hash)


,acquisition_file,zip_bytes,csv_bytes,physical_rows,unique_loans,overlap_with_previous_files,fields_per_row,earliest_report,latest_report,scan_seconds,process_peak_working_set_gib
0,2018Q1,485865913,7061016083,23902313,458830,0,114,201801,202606,267.82,0.1688
1,2018Q2,413171695,5968239705,20310053,454581,0,114,201804,202606,270.28,0.2205
2,2018Q3,418586244,6001364813,20328492,500018,0,114,201807,202606,262.62,0.2898
3,2018Q4,333305218,4715963115,15857103,421040,0,114,201810,202606,236.12,0.2898
4,2019Q1,257294101,3583597065,11962021,341865,0,114,201901,202606,163.52,0.2976


origination_cohort,2018Q1,2018Q2,2018Q3,2018Q4,2018_total
acquisition_file,,,,,
2018Q1,260423,0,0,0,260423
2018Q2,153395,298262,0,0,451657
2018Q3,3494,192765,302953,0,499212
2018Q4,180,3361,171034,245754,420329
2019Q1,259,322,3629,144102,148312


,acquisition_file,check,count
0,2018Q1,loan_first_payment_unresolved,52
1,2018Q1,row_origination_missing_or_invalid,0
2,2018Q1,row_first_payment_missing_or_invalid,0
3,2018Q1,row_report_missing_or_invalid,0
4,2018Q2,loan_first_payment_unresolved,57
5,2018Q2,row_origination_missing_or_invalid,0
6,2018Q2,row_first_payment_missing_or_invalid,0
7,2018Q2,row_report_missing_or_invalid,0
8,2018Q3,loan_first_payment_unresolved,47
9,2018Q3,row_origination_missing_or_invalid,0


Total distinct IDs across files: 2176334
2018 counts remain provisional; assess later acquisitions and release consistency.


,measure,original,current,difference
0,physical_rows,23563399,23902313,338914
1,unique_loans,458830,458830,0
2,fields_per_row,113,114,1


Same decompressed hash as original: False


## 4. Save aggregate review outputs
No loan-level records, credentials, or raw files are included.

In [4]:
out = ROOT / 'data/processed/Fannie_Mae/notebook_18c_pilot_schema_audit' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
out.mkdir(parents=True, exist_ok=False)
inventory_df.to_csv(out / 'archive_inventory.csv', index=False)
cohorts.to_csv(out / 'origination_cohort_counts.csv', index=False)
quality_df.to_csv(out / 'date_quality_checks.csv', index=False)
coverage.to_csv(out / '2018_acquisition_coverage.csv')
reconciliation.to_csv(out / '2018q1_release_reconciliation.csv', index=False)
summary = dict(layout_fields=EXPECTED_FIELDS, source_folder=str(RAW), original_q1_sha256=old_hash, current_q1_sha256=str(q1.decompressed_sha256), files=quarters, distinct_ids_across_files=len(prior_ids), cross_file_overlap_occurrences=int(inventory_df.overlap_with_previous_files.sum()), reviewed_positions={k:positions[k] for k in ['orig','report','first_payment']}, sampling_performed=False, targets_constructed=False, coverage_complete=False, release_consistency_verified=False, semantic_mapping_verified=False)
(out / 'audit_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
review = out / 'notebook_18c_pilot_schema_review.zip'
with zipfile.ZipFile(review, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(out.iterdir()):
        if f.suffix in {'.csv','.json'}: z.write(f, f.name)
print('Upload only:', review)


Upload only: e:\capstone_project\mortgage-model-reliability\data\processed\Fannie_Mae\notebook_18c_pilot_schema_audit\20261125T205656110343Z\notebook_18c_pilot_schema_review.zip
